In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
"""
.. _uoi_lasso:

UoI-Lasso for sparse, minimal bias, regression
=============================r[i================

This example with demonstrate the ability of UoI-Lasso to recover sparse
models with minimal bias.

"""

###############################################################################
# Load synthetic data
# -------------------
#
# The synthetic data will have 40 features, 10 of which are informative and
# 1 response variable.


import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from sklearn.linear_model import LinearRegression, LassoCV

from pyuoi.linear_model import *
from pyuoi.datasets import make_linear_regression
import pandas as pd
from scipy.linalg import solve_discrete_lyapunov
from var_utils import *
from tqdm import tqdm

### Gaussian VAR model

In [4]:
n_features = 10
n_samples = 20
lag = 1


data, transition_matrices, cov = generate_sparse_stationary_var_process(
    n_features,
    n_samples,
    lag=lag,
    sparsity=0.5,
    spectral_radius=0.98,  # Ensures stationarity
    process_type='gaussian'

)

In [5]:
dense_matrices = [M.toarray() for M in  transition_matrices]

# vecortized the ground truth transition matrices
B_truth = np.vstack([m.T for m in dense_matrices]).T.flatten()     
X,Y = vectorization(data, lag)

In [7]:
uoi_var = UoI_Lasso(n_real_features = n_features, fit_VAR = True)
uoi_var.fit(X, Y)
B_model = uoi_var.coef_

In [9]:
selection_accuracy(B_truth, B_model)

0.4714285714285714

In [10]:
# estimation error
est_mask = B_model != 0
np.linalg.norm(B_truth*est_mask - B_model)**2

4.015286355464547

### Poisson VAR model

In [12]:
# Example usage
n_features = 10
n_samples = 10
lag = 1


base_intensity = np.random.randint(1, 10, n_features).astype(np.float64)  # Base rates for each feature

data, transition_matrices, _ = generate_sparse_stationary_var_process(
    n_features,
    n_samples,
    lag=lag,
    sparsity=0.5,
    quench_factor = 1, 
    spectral_radius=0.9,  # Ensures stationarity
    process_type='poisson',
    base_intensity=base_intensity

)

In [13]:
dense_matrices = [M.toarray() for M in  transition_matrices]
# vecortized the ground truth transition matrices
B_truth = np.vstack([m.T for m in dense_matrices]).T.flatten()     
X,Y = vectorization(data, lag)

In [14]:
# have to set fit_intercept = False since the curretn VAR implementation doesn consider constant model terms
poisson = UoI_Poisson(n_real_features = n_features, fit_VAR =True, max_iter=2500, fit_intercept=False)

poisson.fit(X, Y)

/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi

/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi

/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi/src/pyuoi/lbfgs/__init__.py:205: UserWarning: The algorithm routine reaches the maximum number of iterations.
  return opt.minimize(f, x0, progress=progress, args=args)
/Users/yao/packages/pyuoi

AttributeError: 'UoI_Poisson' object has no attribute 'estimation_target'

AttributeError: 'UoI_Poisson' object has no attribute 'estimation_target'

AttributeError: 'UoI_Poisson' object has no attribute 'estimation_target'

In [15]:
B_model = poisson.coef_
B_truth = np.vstack([m.toarray().T for m in transition_matrices]).T.flatten()
selection_accuracy(B_truth, B_model)

0.21311475409836067